In [2]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from datetime import datetime

from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct, WhiteKernel


# =====================
# Config
# =====================
np.random.seed(42)

base_path = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"

file_names = [
    "n_base.csv", "n_base_OH_rebuilt.csv", "n_base_FP_rebuilt.csv",
    "n_all.csv",  "n_all_OH_rebuilt.csv",  "n_all_FP_rebuilt.csv",
    "m_base.csv", "m_base_OH_rebuilt.csv", "m_base_FP_rebuilt.csv",
    "m_all.csv",  "m_all_OH_rebuilt.csv",  "m_all_FP_rebuilt.csv"
]

target_vars = ["Jsc", "Voc", "FF", "PCEmax"]

today = datetime.now().strftime("%Y%m%d")
# 例：あなたの運用に合わせてフォルダ名を明示
output_dir = os.path.join(
    "./",
    f"{today}_Rebuilt_12Files_gaussprLinear_Train_CV10OOF_OOD"
)
os.makedirs(output_dir, exist_ok=True)

OOD_RATIO = 0.1
MIN_SAMPLES = 20
DUP_CORR_TH = 0.99999

# permutation importance（GPRは重いので控えめ推奨）
PERM_REPEATS = 3

# =====================
# Utils
# =====================
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def rpd(y_true, rmse_val):
    s = float(np.std(y_true, ddof=1))
    return float(s / rmse_val) if rmse_val != 0 else np.nan

def create_ood_split(df, feature_cols, ood_ratio=0.1):
    X = df[feature_cols].to_numpy()
    centroid = np.mean(X, axis=0)
    dists = np.linalg.norm(X - centroid, axis=1)
    num_ood = max(1, int(len(df) * ood_ratio))
    ood_idx = np.argsort(dists)[-num_ood:]
    test_df = df.iloc[ood_idx].copy()
    train_df = df.drop(df.index[ood_idx]).copy()
    return train_df, test_df

def remove_zero_var_features(df, cols):
    v = df[cols].var(axis=0, ddof=1)
    keep = v[(v > 0) & (~v.isna())].index.tolist()
    removed = len(cols) - len(keep)
    if removed > 0:
        print(f"  [Cleaning] Removed {removed} zero-variance features")
    return keep

def remove_collinear_features(df, cols, threshold=0.99999):
    if len(cols) < 2:
        return cols
    corr = df[cols].corr().abs().fillna(0)
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if (upper[c] > threshold).any()]
    if len(to_drop) > 0:
        print(f"  [Cleaning] Removed {len(to_drop)} duplicated/colliner features (|r|>{threshold})")
    return [c for c in cols if c not in to_drop]

def scale_neg1_to_1(train_df, test_df, feature_cols):
    scaler = MinMaxScaler(feature_range=(-1, 1))
    Xtr = scaler.fit_transform(train_df[feature_cols])
    Xte = scaler.transform(test_df[feature_cols])
    return scaler, Xtr, Xte

def cv_oof_predictions(model_factory, X, y, n_splits=10, seed=42):
    """
    Returns:
      oof_pred: np.ndarray shape (n_samples,)
      cv_scores: dict with R2/RMSE/MAE
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.full(shape=(len(y),), fill_value=np.nan, dtype=float)

    for fold, (tr, va) in enumerate(kf.split(X), start=1):
        mdl = model_factory()
        mdl.fit(X[tr], y.iloc[tr].to_numpy())
        oof[va] = mdl.predict(X[va])
    # metrics
    r2 = float(r2_score(y, oof))
    rm = rmse(y, oof)
    ma = float(mean_absolute_error(y, oof))
    return oof, {"CV10_R2": r2, "CV10_RMSE": rm, "CV10_MAE": ma, "CV10_RPD": rpd(y, rm)}

# =====================
# Main
# =====================
summary_rows = []
importance_rows = []

print(f"[INFO] --- gaussprLinear (Python) Started: {today} ---")
print(f"[INFO] Input base path: {base_path}")
print(f"[INFO] Output dir     : {output_dir}")

def make_gausspr_linear():
    # linear kernel + noise
    kernel = DotProduct() + WhiteKernel(noise_level=1e-5)
    return GaussianProcessRegressor(
        kernel=kernel,
        alpha=0.0,
        normalize_y=True,
        random_state=42
    )

for file in file_names:
    in_path = os.path.join(base_path, file)
    if not os.path.exists(in_path):
        print(f"[WARN] Skipping: {file} (Not found)")
        continue

    print(f"\n=== Processing gaussprLinear: {file} ===")
    try:
        df_raw = pd.read_csv(in_path, index_col=0)
        if "X" in df_raw.columns:
            df_raw = df_raw.drop(columns=["X"])
    except Exception as e:
        print(f"[ERROR] read_csv failed: {e}")
        continue

    # numeric only + NA drop
    df_num = df_raw.select_dtypes(include=[np.number]).dropna()
    if len(df_num) < MIN_SAMPLES:
        print(f"  [SKIP] too few samples after NA drop: {len(df_num)}")
        continue

    for target in target_vars:
        if target not in df_num.columns:
            continue

        # IMPORTANT: features exclude ONLY current target (not all target_vars)
        feature_cols = [c for c in df_num.columns if c != target]
        feature_cols = remove_zero_var_features(df_num, feature_cols)
        feature_cols = remove_collinear_features(df_num, feature_cols, threshold=DUP_CORR_TH)

        if len(feature_cols) == 0:
            print(f"  [SKIP] no features for target={target}")
            continue

        # OOD split on raw numeric (before scaling)
        train_df, ood_df = create_ood_split(df_num, feature_cols, ood_ratio=OOD_RATIO)
        if len(train_df) < MIN_SAMPLES:
            print(f"  [SKIP] too few train samples after OOD split: {len(train_df)}")
            continue

        # scale
        scaler, X_train, X_ood = scale_neg1_to_1(train_df, ood_df, feature_cols)
        y_train = train_df[target].copy()
        y_ood = ood_df[target].copy()

        # CV10 OoF on TRAIN
        oof_pred, cvm = cv_oof_predictions(make_gausspr_linear, X_train, y_train, n_splits=10, seed=42)

        # fit final on full train
        model = make_gausspr_linear()
        model.fit(X_train, y_train.to_numpy())

        # predictions
        pred_train = model.predict(X_train)
        pred_ood = model.predict(X_ood)

        # metrics
        train_r2 = float(r2_score(y_train, pred_train))
        train_rmse = rmse(y_train, pred_train)
        train_mae = float(mean_absolute_error(y_train, pred_train))

        ood_r2 = float(r2_score(y_ood, pred_ood)) if len(y_ood) >= 2 else np.nan
        ood_rmse = rmse(y_ood, pred_ood)
        ood_mae = float(mean_absolute_error(y_ood, pred_ood))

        # overfit diagnostics (same columns as your PLS summary style)
        delta_r2 = train_r2 - cvm["CV10_R2"]
        delta_rmse = cvm["CV10_RMSE"] - train_rmse
        overfit_flag = bool((delta_r2 > 0.05) or (delta_rmse > 0.1 * train_rmse))
        overfit_label = "Overfit_suspected" if overfit_flag else "OK"

        # summary row
        best_params = {
            "kernel": str(model.kernel_),
            "model": "GaussianProcessRegressor(DotProduct+WhiteKernel)"
        }

        summary_rows.append({
            "File": file,
            "Target": target,
            "Train_R2": train_r2,
            "Train_RMSE": train_rmse,
            "Train_MAE": train_mae,
            "Train_RPD": rpd(y_train, train_rmse),
            "CV10_R2": cvm["CV10_R2"],
            "CV10_RMSE": cvm["CV10_RMSE"],
            "CV10_MAE": cvm["CV10_MAE"],
            "CV10_RPD": cvm["CV10_RPD"],
            "OOD_R2": ood_r2,
            "OOD_RMSE": ood_rmse,
            "OOD_MAE": ood_mae,
            "OOD_RPD": rpd(y_ood, ood_rmse),
            "n_samples": int(len(df_num)),
            "n_features": int(len(feature_cols)),
            "Best_Params": json.dumps(best_params),
            "Overfit_Flag": overfit_flag,
            "Overfit_Label": overfit_label,
            "Delta_R2_TrainMinusCV10": float(delta_r2),
            "Delta_RMSE_CV10MinusTrain": float(delta_rmse),
        })

        # --- Save model bundle ---
        bundle = {
            "model": model,
            "scaler": scaler,
            "feature_cols": feature_cols,
            "target": target,
            "file": file,
        }
        model_name = f"fixed_{today}_gaussprLinear_model_{file.replace('.csv','')}_{target}.joblib"
        joblib.dump(bundle, os.path.join(output_dir, model_name))

        # --- Save pred CSV (keep columns aligned!) ---
        # Train rownames used as SampleID
        pred_df = pd.concat([
            pd.DataFrame({
                "SampleID": train_df.index.astype(str),
                "Observed": y_train.to_numpy(),
                "Predicted": pred_train.astype(float),
                "Type": "Train"
            }),
            pd.DataFrame({
                "SampleID": train_df.index.astype(str),
                "Observed": y_train.to_numpy(),
                "Predicted": oof_pred.astype(float),
                "Type": "CV10_OOF"
            }),
            pd.DataFrame({
                "SampleID": ood_df.index.astype(str),
                "Observed": y_ood.to_numpy(),
                "Predicted": pred_ood.astype(float),
                "Type": "OOD_Test"
            }),
        ], ignore_index=True)

        pred_name = f"fixed_{today}_gaussprLinear_{file.replace('.csv','')}_{target}_pred.csv"
        pred_df.to_csv(os.path.join(output_dir, pred_name), index=False)

        # --- Permutation importance on TRAIN (optional / heavy) ---
        print(f"  Computing permutation importance for {target} (repeats={PERM_REPEATS})...")
        try:
            perm = permutation_importance(
                model, X_train, y_train.to_numpy(),
                n_repeats=PERM_REPEATS, random_state=42, n_jobs=-1,
                scoring="r2"
            )
            order = np.argsort(perm.importances_mean)[::-1]
            for idx in order:
                mean_imp = float(perm.importances_mean[idx])
                if mean_imp > 1e-5:
                    importance_rows.append({
                        "File": file,
                        "Target": target,
                        "Feature": feature_cols[idx],
                        "Importance_Mean": mean_imp,
                        "Importance_Std": float(perm.importances_std[idx]),
                    })
        except Exception as e:
            print(f"  [WARN] permutation importance failed: {e}")

        print(f"  Target={target} | Train_R2={train_r2:.3f} | CV10_R2={cvm['CV10_R2']:.3f} | OOD_R2={ood_r2:.3f}")

# --- Save summary / importance ---
df_sum = pd.DataFrame(summary_rows)
sum_path = os.path.join(output_dir, f"fixed_{today}_gaussprLinear_summary.csv")
df_sum.to_csv(sum_path, index=False)

df_imp = pd.DataFrame(importance_rows)
imp_path = os.path.join(output_dir, f"fixed_{today}_gaussprLinear_feature_importance.csv")
df_imp.to_csv(imp_path, index=False)

print(f"\n[INFO] gaussprLinear Finished.\n[INFO] Summary: {sum_path}\n[INFO] Importance: {imp_path}")


[INFO] --- gaussprLinear (Python) Started: 20251217 ---
[INFO] Input base path: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/
[INFO] Output dir     : ./20251217_Rebuilt_12Files_gaussprLinear_Train_CV10OOF_OOD

=== Processing gaussprLinear: n_base.csv ===
  Computing permutation importance for Jsc (repeats=3)...
  Target=Jsc | Train_R2=0.938 | CV10_R2=0.932 | OOD_R2=0.685
  Computing permutation importance for Voc (repeats=3)...
  Target=Voc | Train_R2=0.697 | CV10_R2=0.676 | OOD_R2=0.734
  Computing permutation importance for FF (repeats=3)...
  Target=FF | Train_R2=0.666 | CV10_R2=0.638 | OOD_R2=0.556
  Computing permutation importance for PCEmax (repeats=3)...
  Target=PCEmax | Train_R2=0.950 | CV10_R2=0.946 | OOD_R2=0.889

=== Processing gaussprLinear: n_base_OH_rebuilt.csv ===
  [Cleaning] Removed 182 zero-variance features
  Computing permutation importance for Jsc (repeats=3)...
  Tar

  [Cleaning] Removed 3 duplicated/colliner features (|r|>0.99999)
  Computing permutation importance for Voc (repeats=3)...
  Target=Voc | Train_R2=0.891 | CV10_R2=0.611 | OOD_R2=-0.013
  [Cleaning] Removed 1 zero-variance features
  [Cleaning] Removed 3 duplicated/colliner features (|r|>0.99999)
  Computing permutation importance for FF (repeats=3)...
  Target=FF | Train_R2=0.939 | CV10_R2=0.726 | OOD_R2=-0.437
  [Cleaning] Removed 1 zero-variance features
  [Cleaning] Removed 3 duplicated/colliner features (|r|>0.99999)
  Computing permutation importance for PCEmax (repeats=3)...
  Target=PCEmax | Train_R2=0.990 | CV10_R2=0.959 | OOD_R2=0.551

=== Processing gaussprLinear: m_all_FP_rebuilt.csv ===
  [Cleaning] Removed 42 duplicated/colliner features (|r|>0.99999)
  Computing permutation importance for Jsc (repeats=3)...
  Target=Jsc | Train_R2=0.968 | CV10_R2=0.932 | OOD_R2=0.294
  [Cleaning] Removed 42 duplicated/colliner features (|r|>0.99999)
  Computing permutation importance for

/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  Target=FF | Train_R2=0.872 | CV10_R2=0.713 | OOD_R2=0.366
  [Cleaning] Removed 42 duplicated/colliner features (|r|>0.99999)
  Computing permutation importance for PCEmax (repeats=3)...
  Target=PCEmax | Train_R2=0.978 | CV10_R2=0.952 | OOD_R2=0.874

[INFO] gaussprLinear Finished.
[INFO] Summary: ./20251217_Rebuilt_12Files_gaussprLinear_Train_CV10OOF_OOD/fixed_20251217_gaussprLinear_summary.csv
[INFO] Importance: ./20251217_Rebuilt_12Files_gaussprLinear_Train_CV10OOF_OOD/fixed_20251217_gaussprLinear_feature_importance.csv
